In [ ]:
import random
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

In [ ]:
#sonuc her zaman aynı cıkması ıcın random u sabıtlıyorum
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [ ]:
# Veri: Wine (178 örnek, 13 özellik, 3 sınıf)
X, y = load_wine(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # sadece train üzerinde fit
X_test = scaler.transform(X_test)

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax"),  #3 sınıf oldugu ıcın softmax kullanıyoruz softmax: 3 çıktıyı toplamı 1 olan olasılıklara çevirir
])
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 32)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 3)              │            51 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,027 (4.01 KB)

 Trainable params: 1,027 (4.01 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.SparseCategoricalCrossentropy(),
    metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
)

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=8,
    verbose=2,
    callbacks=[early_stopping],
)

Epoch 1/25
15/15 - 5s - 324ms/step - accuracy: 0.5310 - loss: 1.0769 - val_accuracy: 0.3793 - val_loss: 1.1887
Epoch 2/25
15/15 - 1s - 35ms/step - accuracy: 0.4956 - loss: 1.0242 - val_accuracy: 0.4828 - val_loss: 1.0422
Epoch 3/25
15/15 - 0s - 26ms/step - accuracy: 0.6018 - loss: 0.9035 - val_accuracy: 0.6207 - val_loss: 0.9182
Epoch 4/25
15/15 - 0s - 27ms/step - accuracy: 0.6726 - loss: 0.8057 - val_accuracy: 0.7241 - val_loss: 0.8101
Epoch 5/25
15/15 - 0s - 18ms/step - accuracy: 0.6903 - loss: 0.7505 - val_accuracy: 0.7931 - val_loss: 0.7136
Epoch 6/25
15/15 - 0s - 16ms/step - accuracy: 0.7699 - loss: 0.6643 - val_accuracy: 0.8966 - val_loss: 0.6372
Epoch 7/25
15/15 - 0s - 18ms/step - accuracy: 0.7876 - loss: 0.5886 - val_accuracy: 0.9310 - val_loss: 0.5521
Epoch 8/25
15/15 - 0s - 14ms/step - accuracy: 0.7876 - loss: 0.5939 - val_accuracy: 1.0000 - val_loss: 0.4646
Epoch 9/25
15/15 - 0s - 18ms/step - accuracy: 0.8142 - loss: 0.5401 - val_accuracy: 1.0000 - val_loss: 0.3890
Epoch 10/

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f} | Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print(classification_report(y_test, y_pred, target_names=load_wine().target_names))

Test loss: 0.1135 | Test accuracy: 0.9444
              precision    recall  f1-score   support

     class_0       0.92      1.00      0.96        12
     class_1       0.93      0.93      0.93        14
     class_2       1.00      0.90      0.95        10

    accuracy                           0.94        36
   macro avg       0.95      0.94      0.95        36
weighted avg       0.95      0.94      0.94        36

